# 06.8 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/08-anomaly-detection/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## Exercise 1

Sorted orders: 38, 42, 44, 47, **49, 52**, 55, 58, 61, 410. Median = (49 + 52)/2 = 50.5.

Absolute deviations from 50.5: 12.5, 8.5, 6.5, 3.5, 1.5, 1.5, 4.5, 7.5, 10.5, 359.5. Sorted: 1.5, 1.5, 3.5, 4.5, **6.5, 7.5**, 8.5, 10.5, 12.5, 359.5, so MAD = 7.0.

For £410: M = 0.6745 × (410 − 50.5) / 7 = 0.6745 × 51.36 = **34.6**, far above 3.5. Every other order has |M| ≤ 1.2. Median and MAD are not dragged by the outlier, so there is no masking.

In [2]:
orders = np.array([42, 55, 38, 61, 47, 52, 44, 58, 49, 410], dtype=float)   # order values, £

med = np.median(orders)
mad = np.median(np.abs(orders - med))
M = 0.6745 * (orders - med) / mad
print(f"median {med}, MAD {mad}")
print("modified z:", M.round(2))

median 50.5, MAD 7.0
modified z: [-0.82  0.43 -1.2   1.01 -0.34  0.14 -0.63  0.72 -0.14 34.64]


## Exercise 2

In [3]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

products = sales[~sales["StockCode"].isin(["M", "POST", "DOT", "C2", "BANK CHARGES", "AMAZONFEE", "D", "CRUK", "PADS"])]
print(f"removed {len(sales) - len(products):,} non-product lines")
sales = products

392,693 clean transaction lines, 4,338 customers
removed 1,543 non-product lines


In [4]:
inv = sales.groupby("Invoice").agg(
    customer=("Customer ID", "first"), country=("Country", "first"),
    when=("InvoiceDate", "first"), amount=("Revenue", "sum"),
    items=("Quantity", "sum"), lines=("StockCode", "nunique"), max_price=("Price", "max"))
inv["hour"] = inv["when"].dt.hour
inv["avg_price"] = inv["amount"] / inv["items"]
print(f"{len(inv):,} invoices")

from sklearn.ensemble import IsolationForest
features = ["amount", "items", "avg_price", "hour"]
X_inv = np.log1p(inv[features])                  # log: amounts and items are very skewed
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42).fit(X_inv)
inv["score"] = -iso.score_samples(X_inv)          # higher = easier to isolate = odder
inv["flag"] = iso.predict(X_inv) == -1
first_line = sales.groupby("Invoice").agg(code=("StockCode", "first"), description=("Description", "first"))
inv = inv.join(first_line)
print(f"flagged {inv['flag'].sum()} of {len(inv):,} invoices")

inv.sort_values("score", ascending=False).head(10)[["amount", "items", "lines", "avg_price", "hour", "description", "score"]].round(2)

18,402 invoices


flagged 185 of 18,402 invoices


,amount,items,lines,avg_price,hour,description,score
Invoice,,,,,,,
577130,165.00,1,1,165.00,19,RUSTIC SEVENTEEN DRAWER SIDEBOARD,0.75
536835,295.00,1,1,295.00,18,VINTAGE RED KITCHEN CABINET,0.74
545900,195.00,1,1,195.00,17,LOVE SEAT ANTIQUE WHITE METAL,0.73
556266,8.50,1,1,8.50,19,BLACK KITCHEN SCALES,0.72
556446,649.50,1,1,649.50,15,PICNIC BASKET WICKER 60 PIECES,0.72
556267,3027.31,9811,31,0.31,19,CLASSIC FRENCH STYLE BASKET NATURAL,0.72
561680,25.50,2,1,12.75,19,SET OF 16 VINTAGE ROSE CUTLERY,0.72
573161,39.95,1,1,39.95,9,SET/4 WHITE RETRO STORAGE CUBES,0.71
545902,250.00,2,2,125.00,17,VINTAGE RED KITCHEN CABINET,0.71


With the adjustments gone, the top of the list is single-item orders of expensive furniture and a few orders at unusual hours. These are still not fraud, but they are now genuine *customer* behaviour worth a look, which is what we wanted the detector to find. Cleaning first makes anomaly detection far more useful.

## Exercise 3

In [5]:
flagged = inv[inv["flag"]]
comparison = pd.DataFrame({"flagged median": flagged[features].median(),
                           "all invoices median": inv[features].median()})
comparison["ratio"] = comparison["flagged median"] / comparison["all invoices median"]
comparison.round(2)

,flagged median,all invoices median,ratio
amount,21.0,301.65,0.07
items,2.0,156.00,0.01
avg_price,8.5,1.87,4.55
hour,14.0,12.00,1.17


`items` differs by far the most (ratio 0.01), then `avg_price` and `amount`. A typical flagged invoice (after cleaning) is tiny: 2 items for about £21, at £8.50 each. A typical invoice is 156 items for about £302, at £1.87 each. Hour of day barely differs (14:00 vs 12:00). ShopWave is mostly a wholesaler of cheap gifts, so in this data "odd" mostly means "a few items, bought like a retail customer, at a high price each".